TODO: natively vectorize as many functions as possible to avoid iterations and speed up computation.

In [1]:
from engine import WordleEngine, Square
import numpy as np
from wordfreq import word_frequency

WORD_LEN = 5
LOCALE = 'en'
FREQ_VEC = np.vectorize(lambda word: word_frequency(word, LOCALE, minimum=1e-8), otypes=[float])

engine = WordleEngine('words.txt', WORD_LEN)
df = engine._df
df

,0,1,2,3,4
aahed,97,97,104,101,100
aalii,97,97,108,105,105
aapas,97,97,112,97,115
aargh,97,97,114,103,104
aarti,97,97,114,116,105
...,...,...,...,...,...
zuzim,122,117,122,105,109
zygal,122,121,103,97,108
zygon,122,121,103,111,110
zymes,122,121,109,101,115


Play a game of Worlde below. Record your guesses and responses as you make them and rerun the following cells to refresh the suggestions.

In [2]:
def convert_str_to_squares(abbrev: str) -> np.ndarray:
    """Convenience wrapper to save myself some copy pasting."""
    squares = []
    for letter in abbrev:
        assert letter in ('g', 'y', 'b')
        if letter == 'g':
            squares.append(Square.GREEN.value)
        elif letter == 'y':
            squares.append(Square.YELLOW.value)
        else:
            squares.append(Square.BLACK.value)
    return np.array(squares, dtype=np.uint8)

In [3]:
guesses = [
    'salet',
    'adieu',
    'lover',
    'novel'
]
responses = [
    'bbygb',
    'bbbgb',
    'ygggb',
    'ggggg'
]

Iteratively filter the full list of words based on guesses and their square responses until we arrive at a subset of candidates. We can sort our suggestions to give the most likely candidates by using word frequency.

In [4]:
filtered = df.copy()
filtered['freq'] = FREQ_VEC(filtered.index)
for guess, response in zip(guesses, responses):
    squares = convert_str_to_squares(response)
    gs_np = df.loc[guess].to_numpy()
    mask = [engine.is_match(gs_np, row.drop('freq').to_numpy(), squares) \
            for _, row in filtered.iterrows()]
    filtered = filtered[mask]
    reccs = filtered.sort_values(by='freq', ascending=False)
    print(reccs[:5], '=' * 35, sep='\n')

         0    1    2    3    4      freq
level  108  101  118  101  108  0.000257
model  109  111  100  101  108  0.000135
lower  108  111  119  101  114  0.000129
loved  108  111  118  101  100  0.000100
older  111  108  100  101  114  0.000083
         0    1    2    3    4      freq
level  108  101  118  101  108  0.000257
lower  108  111  119  101  114  0.000129
novel  110  111  118  101  108  0.000043
wheel  119  104  101  101  108  0.000032
lover  108  111  118  101  114  0.000018
         0    1    2    3    4          freq
novel  110  111  118  101  108  4.270000e-05
hovel  104  111  118  101  108  2.040000e-07
         0    1    2    3    4      freq
novel  110  111  118  101  108  0.000043


Once you have the solution, visually ascertain that the produced squares match the game and validate our own function.

In [5]:
expected = 'novel'

for guess, response in zip(guesses, responses):
    gs_np = df.loc[guess].to_numpy()
    exp_np = df.loc[expected].to_numpy()
    squares = engine.wordle_compare(gs_np, exp_np)
    print(guess, engine.render(squares))
    assert np.all(squares == convert_str_to_squares(response))
    assert engine.is_match(gs_np, exp_np, squares)

salet ⬛⬛🟨🟩⬛
adieu ⬛⬛⬛🟩⬛
lover 🟨🟩🟩🟩⬛
novel 🟩🟩🟩🟩🟩


Calculate the entropy of each possible guess and square permutation. This will be stored in an array of shape (`NUM_WORDS` , $3^N$) where $N$ = `WORD_LEN`. We will assume a uniform probability distribution over the possible words. The probability $P(s | w)$ of getting square pattern $s$ after guessing word $w \in W$ is simply the proportion of words $c$ in the corpus such that `is_match(w, c, s)`. We define the information gained by this guess (measured in bits) as:

```math
I(w, s) = - \log_2 P(s | w) = - \log_2 \left( \frac{\text{matches}(w, s)}{|W|} \right)
```

We want to choose the word that maximizes the expected information gain. That is, the entropy $H$ given by the following sum over all square patterns:

```math
H(w) = - \sum_s P(s | w) \cdot \log_2 P(s | w)
```